# Clase 7 — Attention: cada palabra arma su propio resumen del contexto

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Cuando leés «lo» al final de una oración larga,
> saltás para atrás y encontrás a qué se refiere sin releer todo. Attention es eso,
> escrito como una cuenta: **cada palabra reparte 100 puntos de atención entre las
> demás y se arma un promedio ponderado del contexto**.

---

**Cuando termines este notebook vas a poder:**

1. Escribir attention completa en doce líneas, sin usar ninguna función mágica.
2. Leer una matriz de atención sabiendo **qué suma 1 y qué no**.
3. Explicar para qué está la máscara causal, y mostrar con números qué pasa si la
   aplicás en el momento equivocado.
4. Armar un **bloque transformer entero** y verificar que las formas cierran.
5. Explicar por qué un mapa de atención **no es** una explicación del modelo.

**Lo que hace falta traer:** la clase 6 (embeddings). Nada más.

> **Hoy construimos, no entrenamos.** Al final de este notebook vas a tener un
> transformer completo y **apagado**. Encenderlo es la clase 8, y esa espera es a
> propósito: si entrenáramos hoy, el entusiasmo se comería el mecanismo.

**Cómo se usa este notebook.** Corré las celdas en orden (`Shift + Enter`). Donde diga
`# TODO` te toca a vos: las celdas ya traen una versión que funciona, y el ejercicio es
**cambiarla y mirar qué pasa**. Las verificaciones son amables: te dicen qué salió y qué
no, sin tirarte un stack trace por la cabeza.

> Si corrés todo de una sola vez vas a ver un par de `OJO ·`. **No están rotas**: son los
> ejercicios esperando que los hagas. Cuando los resuelvas, pasan a `OK ·`.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el repo y deja
todo a mano; en tu máquina no hace prácticamente nada porque ya tenés los archivos.
Corrémela y seguí de largo: no hay nada que entender acá.

In [ ]:
# =============================================================================

# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.

# En Colab: clona el repo del curso e instala lo que falte.

# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.

# =============================================================================

import os

import pathlib

import subprocess

import sys



REPO = "https://github.com/johannperez/dlss"

RAMA = "claude/dl-course-social-scientists-279dd2"

CARPETA = "dlss"





def en_colab():

    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""

    try:

        import google.colab  # noqa: F401

        return True

    except ImportError:

        return False





def buscar_raiz(desde=None):

    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""

    punto = pathlib.Path(desde or os.getcwd()).resolve()

    for candidata in [punto, *punto.parents]:

        if (candidata / "_shared" / "ssdl").is_dir():

            return candidata

    return None





raiz = buscar_raiz()



if raiz is None and en_colab():

    if not pathlib.Path(CARPETA).exists():

        print("Bajando el repo del curso (una sola vez por sesión)...")

        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,

                        REPO, CARPETA], check=True)

    raiz = buscar_raiz(CARPETA)



if raiz is None:

    raise SystemExit(

        "No encuentro el repo del curso.\n"

        "· En Colab: revisá que la celda haya podido clonar (necesita internet).\n"

        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`."

    )



# Dependencias: en Colab casi todas vienen puestas. Instalamos sólo lo que falte.

for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),

                        ("torch", "torch")]:

    try:

        __import__(modulo)

    except ImportError:

        print(f"Instalando {paquete}...")

        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],

                       check=True)



sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`

os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos



print(f"Listo. Repo del curso en: {raiz}")

### Imports, semilla y una función de verificación

Fijamos la semilla en 1983 **siempre**. Acá importa especialmente: casi todo lo de hoy usa
matrices al azar (todavía no entrenamos nada), y sin semilla fija los números que ves vos
y los que ve la persona de al lado no coinciden, y la clase se vuelve una discusión sobre
ruido.

In [ ]:
import json
import pathlib

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()


def revisar(condicion, si_va, si_no):
    """Verificación amable: te dice qué pasó, no te tira un stack trace encima."""
    print(("OK   · " + si_va) if condicion else ("OJO  · " + si_no))
    return bool(condicion)

---

## 1. La idea

Un modelo que lee palabra por palabra arrastrando una mochila de memoria (la RNN de la
clase 6) **se olvida del principio**. Attention hace lo contrario: no arrastra nada.

> Cada palabra mira a todas las demás **al mismo tiempo**, le pone un peso a cada una, y
> se arma un resumen del contexto donde cada palabra pesa distinto. Los pesos suman 1.

**La analogía de ciencias sociales.** Es un **promedio ponderado**, la misma cuenta que
hacés cuando el parcial vale 60 y el práctico 40. Y la matriz de pesos que vamos a
construir es, tal cual, una **matriz de adyacencia dirigida**: quién mira a quién, con
pesos. Si viste análisis de redes, ya conocés el objeto — incluida la asimetría (que A
cite a B no implica que B cite a A).

**Lo que NO es.** No es una memoria. No hay ningún estado que se arrastre de palabra en
palabra. Todas las palabras se miran **en paralelo, de una**. Eso es exactamente lo que
permite entrenarlo rápido en GPU, y también la razón de que exista una *ventana de
contexto* finita: si cada palabra mira a todas, el costo crece con el **cuadrado** de la
longitud. Volvemos sobre esto en la clase 8.

---

## 2. Miralo: el promedio ponderado, con tres vectores y tres pesos

Antes de escribir attention, hagamos a mano la única cuenta que hay adentro.

Tres palabras del contexto, cada una con un vector de dos números (dos, para poder
dibujarlas como flechas). Y tres pesos que suman 1.

> **Apostá primero.** Antes de correr: si el peso más grande se lo lleva `proyecto`,
> ¿dónde va a caer la flecha del resultado? ¿Cerca de `proyecto`, en el medio de las tres,
> o fuera de la nube?

In [ ]:
# Las tres palabras del contexto, con dos números cada una.
# (Un modelo de verdad usa 100 o 768 números por palabra; acá usamos 2 para dibujar.)
contexto = {
    "proyecto": np.array([4.0,  1.0]),
    "comisión": np.array([1.0,  4.0]),
    "ministro": np.array([-2.0, 2.0]),
}

# Los pesos de atención: cuánto le importa a la palabra "lo" cada una de las tres.
pesos = np.array([0.60, 0.30, 0.10])

print("¿Suman 1 los pesos?", pesos.sum())

# EL PROMEDIO PONDERADO. Toda la cuenta de attention es esta línea.
matriz_contexto = np.stack(list(contexto.values()))     # (3 palabras, 2 números)
resumen = pesos @ matriz_contexto                       # (2 números)

print("formas:", pesos.shape, "@", matriz_contexto.shape, "->", resumen.shape)
print("el resumen que se arma «lo»:", resumen)

In [ ]:
# Y lo dibujamos: las tres flechas del contexto, y en violeta el promedio ponderado.
fig, ax = g.figura(6.4, 5.2, titulo="El resumen cae ADENTRO de la nube, nunca afuera",
                   xlabel="primer número", ylabel="segundo número")

for (palabra, v), w in zip(contexto.items(), pesos):
    ax.arrow(0, 0, v[0], v[1], head_width=0.18, length_includes_head=True,
             color=g.DATOS, alpha=0.35 + w, linewidth=1 + 4 * w)
    ax.text(v[0] * 1.06, v[1] * 1.06, f"{palabra}  ×{w:.2f}", color=g.DATOS, fontsize=10)

ax.arrow(0, 0, resumen[0], resumen[1], head_width=0.22, length_includes_head=True,
         color=g.PARAMETRO, linewidth=3)
ax.text(resumen[0] * 1.05, resumen[1] * 1.05 - 0.35, "el resumen de «lo»",
        color=g.PARAMETRO, fontsize=11, weight="bold")
ax.set_xlim(-3, 5.4); ax.set_ylim(-0.6, 4.8)
ax.axhline(0, color=g.C["borde"], lw=1); ax.axvline(0, color=g.C["borde"], lw=1)
plt.show()

### `TODO 1` — que el resumen quede casi encima de «proyecto»

Cambiá los tres pesos de la celda de abajo para que la flecha violeta quede prácticamente
encima de la flecha de `proyecto`.

**Pista 1.** Los pesos tienen que seguir sumando 1. No hay manera de que attention le dé
más a una palabra sin sacarle a otra: eso es *repartir*.

**Pista 2.** Si le das 1,0 a `proyecto` y 0 a las otras dos, el resumen **es** `proyecto`.
Probá primero eso, y después bajá un poquito hasta 0,9 y mirá cuánto se movió.

**Cómo saber si te salió.** La distancia entre el resumen y `proyecto` tiene que quedar
por debajo de 0,5. La celda te lo dice.

In [ ]:
# TODO 1: cambiá estos tres números (tienen que seguir sumando 1)
mis_pesos = np.array([0.60, 0.30, 0.10])

resumen_2 = mis_pesos @ matriz_contexto
distancia = np.linalg.norm(resumen_2 - contexto["proyecto"])

print(f"tus pesos suman {mis_pesos.sum():.2f}")
print(f"el resumen te dio {resumen_2.round(2)}  ·  «proyecto» está en "
      f"{contexto['proyecto'].round(2)}")
print(f"distancia entre los dos: {distancia:.2f}")

revisar(abs(mis_pesos.sum() - 1) < 1e-6,
        "los pesos suman 1: es un reparto de verdad.",
        f"los pesos suman {mis_pesos.sum():.2f} y tienen que sumar 1. "
        "Si suman más, el resumen se va a escapar de la nube.")
revisar(distancia < 0.5,
        "el resumen quedó pegado a «proyecto». Eso es prestarle casi toda la atención.",
        "todavía está lejos. Subile el peso a «proyecto» y bajáselo a las otras dos.")

---

## 3. Attention, paso a paso

Ya sabés hacer el promedio ponderado. Lo único que falta es **de dónde salen los pesos**.

Vamos a trabajar con una oración de verdad del corpus de discursos de apertura:

> *«si tenemos que corregir algo, lo vamos a corregir»*

y con los **embeddings que entrenaste en la clase 6** (coocurrencia → PPMI → SVD, con
numpy y nada más). Los volvemos a calcular acá: tarda entre cinco y treinta segundos según
la máquina, y no necesita internet.

In [ ]:
# Los embeddings de la clase 6, con la MISMA configuración de aquel notebook.
# Sale del corpus real de discursos de apertura (1984-2019). Tarda ~25 s.
with utils.medir_tiempo("calcular los embeddings de la clase 6"):
    V_emb, palabras = datos.embeddings_ppmi_svd(
        datos.corpus_mini_gpt(), dim=100, top=2000, ventana=5, minimo=8)

indice = {p: i for i, p in enumerate(palabras)}
print(f"{V_emb.shape[0]} palabras, cada una con {V_emb.shape[1]} números.")

# Un recordatorio de la clase 6 de que esto no es magia: son vecinos de uso.
print("vecinas de «educación»:",
      [p for p, _ in datos.vecinos("educación", V_emb, palabras, k=5)])

In [ ]:
oracion = "si tenemos que corregir algo lo vamos a corregir".split()

# X es la oración convertida en números: una fila por palabra.
# Si alguna palabra no estuviera en el vocabulario, le damos un vector al azar y avisamos.
filas = []
for palabra in oracion:
    if palabra in indice:
        filas.append(V_emb[indice[palabra]])
    else:
        print(f"OJO · «{palabra}» no está en el vocabulario; le pongo un vector al azar.")
        filas.append(np.random.randn(V_emb.shape[1]) * 0.1)

X = torch.tensor(np.stack(filas), dtype=torch.float32)

# EL HÁBITO DE LA CLASE: imprimir la forma después de cada paso.
# No es manía. Es el 80% de los errores de esta materia, atrapados a tiempo.
print("X.shape =", tuple(X.shape), " -> (palabras, números por palabra)")

### Las tres cajas de perillas

`W_q`, `W_k` y `W_v` son las tres matrices que se aprenden. Hoy las llenamos **al azar**,
porque hoy no entrenamos: queremos ver el mecanismo, no el resultado.

Fijate en la celda que las tres se calculan **a partir de la misma `X`**. Q, K y V no son
tres entradas distintas: son tres transformaciones de la misma palabra.

In [ ]:
d_modelo = X.shape[1]     # 100: los números que trae cada palabra
d_cabeza = 16             # los números que usa esta cabeza para buscar y ofrecer

# La escala importa, y vale la pena entender por qué. Los embeddings de la clase 6
# vienen normalizados a norma 1, así que sus números son chiquititos. Si además las
# matrices fueran chiquitas, TODOS los puntajes darían casi cero, el softmax
# repartiría parejo y el mapa saldría gris parejo.
# (Probalo: cambiá 1.5 por 0.1 y volvé a correr hasta el heatmap. Es la mejor
#  manera de entender para qué está el √d de la fórmula.)
ESCALA = 1.5
W_q = torch.randn(d_modelo, d_cabeza) * ESCALA   # cómo se arma "lo que busco"
W_k = torch.randn(d_modelo, d_cabeza) * ESCALA   # cómo se arma "el aviso que publico"
W_v = torch.randn(d_modelo, d_cabeza) * ESCALA   # cómo se arma "lo que aporto"

Q = X @ W_q     # una búsqueda por palabra
K = X @ W_k     # un aviso por palabra
V = X @ W_v     # una mercadería por palabra

print("X.shape =", tuple(X.shape))
print("Q.shape =", tuple(Q.shape), " K.shape =", tuple(K.shape),
      " V.shape =", tuple(V.shape))

### Los puntajes: cada búsqueda contra cada aviso

`Q @ K.T` compara **todas** las búsquedas con **todos** los avisos de una sola vez. El
resultado es una matriz cuadrada: tantas filas como palabras, tantas columnas como
palabras.

Dividimos por `√d`. Es lo único de la fórmula que no tiene una historia linda: con
vectores largos los productos punto se hacen enormes, el softmax se satura (todo 1 y 0) y
el gradiente se muere. Se dividen y listo.

In [ ]:
puntajes = Q @ K.T / d_cabeza ** 0.5

print("puntajes.shape =", tuple(puntajes.shape),
      " -> (quién pregunta, a quién mira)")
print(f"van de {puntajes.min():.2f} a {puntajes.max():.2f}: "
      "números feos, positivos y negativos, que todavía no son porcentajes.")

# El "antes": azul negativo, blanco cero, rojo positivo (el mismo criterio que
# los filtros de la clase 5).
lim = float(puntajes.abs().max())
fig, ax = g.figura(6.6, 5.4, titulo="ANTES del softmax: puntajes crudos")
im = ax.imshow(puntajes.numpy(), cmap=g.CMAPS["filtro"], vmin=-lim, vmax=lim)
ax.set_xticks(range(len(oracion)), oracion, rotation=52, ha="right")
ax.set_yticks(range(len(oracion)), oracion)
ax.set_xlabel("...mira a esta palabra"); ax.set_ylabel("esta palabra...")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
plt.show()

In [ ]:
# El softmax convierte cada FILA en porcentajes que suman 100.
pesos_at = torch.softmax(puntajes, dim=-1)      # dim=-1 -> por filas

g.mapa_de_atencion(pesos_at, oracion,
                   titulo="DESPUÉS del softmax: ahora cada fila es un reparto")
plt.show()

sumas_fila = pesos_at.sum(dim=1)
print("suma de cada fila:", sumas_fila.numpy().round(3))
revisar(torch.allclose(sumas_fila, torch.ones_like(sumas_fila), atol=1e-5),
        "todas las filas suman 1. El softmax hizo su trabajo.",
        "alguna fila no suma 1. Casi seguro le pasaste dim=0 al softmax en vez de dim=-1: "
        "eso normaliza por columnas y rompe el promedio ponderado.")

### `TODO 2` — ¿y las columnas?

Calculá la suma de cada **columna** y mirá qué da. Después contestá abajo, con tus
palabras, por qué existe esa asimetría.

**Pista.** Cada palabra reparte *su propia* atención. Nada obliga a que una palabra sea
mirada, en total, un 100%. Puede ser mirada por todo el mundo, o por nadie.

In [ ]:
# TODO 2: sumá por columnas en vez de por filas. (Cambiá el dim.)
sumas_columna = pesos_at.sum(dim=1)     # <- TODO: esto todavía suma por filas

print("suma de cada columna:", sumas_columna.numpy().round(2))
for palabra, s in zip(oracion, sumas_columna.tolist()):
    print(f"  a «{palabra}» la miran, en total, {s:.2f}")

revisar(not torch.allclose(sumas_columna, torch.ones_like(sumas_columna), atol=1e-3),
        "las columnas NO suman 1, y eso está bien: es la asimetría del mecanismo.",
        "te dieron todas 1: seguís sumando por filas. Probá dim=0.")

**Tu respuesta:** _(escribí acá por qué las filas suman 1 y las columnas no)_

In [ ]:
# Y ahora sí: el promedio ponderado, con esos pesos.
salida = pesos_at @ V

print("pesos_at.shape =", tuple(pesos_at.shape), "@ V.shape =", tuple(V.shape),
      "->", tuple(salida.shape))
print()
print("Entró una oración de", X.shape[0], "palabras y salió una oración de",
      salida.shape[0], "palabras.")
print("Mismo número de palabras, contenido distinto: cada una ahora está")
print("«contextualizada», es decir, se mezcló con las que le importan.")

---

## 4. La máscara causal: no se espía el futuro

Si el modelo tiene que adivinar la palabra 5 y puede mirar la palabra 6, no aprende a
predecir: **aprende a copiar**. Y su loss de entrenamiento va a dar hermosa mientras el
modelo no sirve para nada.

La solución es tapar el triángulo del futuro **antes** del softmax, poniéndole un puntaje
tan bajo (`-inf`) que después del softmax queda exactamente en cero.

> El `-inf` es el detalle que más confunde. Pensalo así: el softmax hace `e` elevado al
> puntaje. `e` elevado a menos infinito es cero. No estamos borrando la celda: le estamos
> poniendo el peor puntaje posible.

In [ ]:
n = len(oracion)

# torch.tril deja el triángulo de abajo (el pasado) y pone cero arriba (el futuro).
permitido = torch.tril(torch.ones(n, n))
print("la máscara (1 = puedo mirar, 0 = todavía no pasó):")
print(permitido.numpy().astype(int))

puntajes_tapados = puntajes.masked_fill(permitido == 0, float("-inf"))
pesos_causales = torch.softmax(puntajes_tapados, dim=-1)

g.mapa_de_atencion(pesos_causales, oracion, mascara_causal=True,
                   titulo="Con máscara causal: cada palabra sólo mira hacia atrás")
plt.show()

print("¿siguen sumando 1 las filas?",
      pesos_causales.sum(dim=1).numpy().round(3))
print("La primera fila es 1,00 en una sola celda: la primera palabra")
print("no tiene a nadie a quien mirar más que a sí misma.")

### `TODO 3` — ¿y si tapo *después* del softmax?

Es un error real y frecuente: primero se hace el softmax y después se ponen en cero las
celdas del futuro. Probalo en la celda de abajo y mirá las sumas de las filas.

**Pista.** No van a sumar 1. Y eso rompe el promedio ponderado: si los pesos suman 0,4, el
resumen que se arma esa palabra es un 60% más chico de lo que debería, sin que nadie avise.

**Cómo saber si te salió.** La celda imprime las sumas. Tienen que dar cualquier cosa
menos 1.

In [ ]:
# TODO 3: acá está el error a propósito. Corré la celda y mirá las sumas.
mal_pesos = torch.softmax(puntajes, dim=-1)      # 1) primero el softmax (con el futuro)
mal_pesos = mal_pesos * permitido                # 2) y recién después tapamos. MAL.

sumas_mal = mal_pesos.sum(dim=1)
print("sumas de las filas cuando tapás DESPUÉS:", sumas_mal.numpy().round(3))
print("sumas de las filas cuando tapás ANTES:  ",
      pesos_causales.sum(dim=1).numpy().round(3))

revisar(not torch.allclose(sumas_mal, torch.ones_like(sumas_mal), atol=1e-3),
        "confirmado: tapando después, las filas dejan de sumar 1 y el promedio "
        "ponderado se desinfla.",
        "te dieron todas 1: fijate que estés multiplicando por la máscara DESPUÉS "
        "del softmax.")

# TODO 3 (segunda parte): ¿cuánta atención se perdió en la peor fila?
perdida = 1 - sumas_mal.min().item()
print(f"\nLa fila más perjudicada perdió el {perdida*100:.0f}% de su atención.")

---

## 5. Attention entera, en una función

Todo lo que hicimos, junto. Doce líneas. **Esta función es attention**: no hay nada más
adentro de un transformer que esto, repetido.

In [ ]:
def atencion(X, W_q, W_k, W_v, causal=True):
    """Attention entera. Doce líneas y ninguna función mágica."""
    Q = X @ W_q                                    # lo que busca cada palabra
    K = X @ W_k                                    # el aviso que publica cada palabra
    V = X @ W_v                                    # lo que aporta cada palabra
    d = Q.shape[-1]
    puntajes = Q @ K.T / d ** 0.5                  # cada búsqueda contra cada aviso
    if causal:                                     # tapar el futuro ANTES del softmax
        futuro = torch.triu(torch.ones_like(puntajes), diagonal=1).bool()
        puntajes = puntajes.masked_fill(futuro, float("-inf"))
    pesos = torch.softmax(puntajes, dim=-1)        # puntajes -> porcentajes que suman 1
    salida = pesos @ V                             # el promedio ponderado de la sección 2
    return salida, pesos


salida_f, pesos_f = atencion(X, W_q, W_k, W_v)
print("salida:", tuple(salida_f.shape), " pesos:", tuple(pesos_f.shape))
revisar(torch.allclose(pesos_f, pesos_causales, atol=1e-6),
        "la función da exactamente lo mismo que los pasos sueltos de arriba.",
        "algo cambió respecto de los pasos sueltos. Revisá el orden: máscara, "
        "después softmax.")

---

## 6. Multi-head: hacer lo mismo cuatro veces, con otras perillas

Una sola cabeza tiene que decidir una sola cosa: a quién mirar. Pero hay muchas maneras de
que una palabra «importe» (es el sujeto, habla del mismo tema, está al lado).

**Multi-head es literalmente eso: llamar a la misma función cuatro veces con matrices
distintas y pegar las salidas una al lado de la otra.** No hay ninguna idea nueva.

In [ ]:
n_cabezas = 4
cabezas, mapas = [], []

for h in range(n_cabezas):
    Wq_h = torch.randn(d_modelo, d_cabeza) * ESCALA
    Wk_h = torch.randn(d_modelo, d_cabeza) * ESCALA
    Wv_h = torch.randn(d_modelo, d_cabeza) * ESCALA
    sal_h, pes_h = atencion(X, Wq_h, Wk_h, Wv_h)
    cabezas.append(sal_h)
    mapas.append(pes_h)

# "Concatenar" = pegar una al lado de la otra. NO promediar.
multi = torch.cat(cabezas, dim=-1)
print(f"{n_cabezas} cabezas de {d_cabeza} números -> {tuple(multi.shape)}")
print("(cada palabra sale con", multi.shape[1], "números: los de las cuatro cabezas juntos)")

fig, ejes = g.figura(13, 11, filas=2, columnas=2,
                     titulo="Cuatro cabezas SIN ENTRENAR sobre la misma oración")
for h, ax in enumerate(ejes.ravel()):
    g.mapa_de_atencion(mapas[h], oracion, ax=ax, mascara_causal=True,
                       titulo=f"cabeza {h + 1}", anotar=False)
plt.show()

> **Ojo con esta figura.** Los cuatro mapas son **distintos entre sí**, y eso ya dice algo:
> cuatro juegos de perillas distintos producen cuatro lecturas distintas del mismo texto.
> Pero son distintos **al azar**: estas cabezas están **sin entrenar**, sus matrices son
> ruido, y ninguno de esos patrones significa nada. En la sección 9 vamos a mirar las
> cabezas de un modelo **entrenado de verdad** sobre el corpus. La comparación entre las
> dos figuras es el mejor argumento de que los patrones **se aprenden** y no vienen de
> fábrica.

---

## 7. Sin positional encoding, volvemos a la bolsa de palabras

Miralo bien: en la función `atencion` **no hay ningún lugar donde entre el orden**. Cada
palabra mira a todas y a todas les hace la misma cuenta.

Vamos a demostrarlo: barajamos la oración, corremos attention, y comparamos.

> **Un detalle honesto.** Para esta demostración apagamos la máscara causal
> (`causal=False`). ¿Por qué? Porque **la máscara sí sabe del orden**: es la única parte de
> attention que lo sabe, y taparía justo lo que queremos mostrar. Lo que queremos ver es
> que **la cuenta de attention, por sí sola, es ciega al orden**.

In [ ]:
# Una permutación fija, para que a todos nos dé lo mismo.
perm = torch.tensor([3, 0, 6, 1, 8, 2, 5, 7, 4])
print("oración original :", " ".join(oracion))
print("oración barajada :", " ".join(oracion[i] for i in perm.tolist()))

sal_orig, _ = atencion(X, W_q, W_k, W_v, causal=False)
sal_perm, _ = atencion(X[perm], W_q, W_k, W_v, causal=False)

# Si attention fuera ciega al orden, la salida de la oración barajada tendría que ser
# la salida original, barajada de la misma manera.
iguales = torch.allclose(sal_perm, sal_orig[perm], atol=1e-5)
print("\ndiferencia máxima:", float((sal_perm - sal_orig[perm]).abs().max()))
revisar(iguales,
        "son exactamente la misma salida, sólo que reordenada. Attention sola NO sabe "
        "qué viene antes: es la bolsa de palabras de la clase 6, pero carísima.",
        "algo no cerró. Fijate que hayas puesto causal=False en las dos llamadas.")

In [ ]:
# La solución: sumarle a cada palabra un vector que dice en qué posición está.
# Usamos posiciones APRENDIDAS (una tabla, una fila por posición). Son más simples de
# explicar que los senos y cosenos del paper de 2017, y es lo que usan hoy casi todos.
emb_posicion = nn.Embedding(32, d_modelo)          # hasta 32 posiciones
with torch.no_grad():
    emb_posicion.weight.normal_(0, 0.4)            # al azar: todavía no entrenamos

posiciones = torch.arange(n)
X_con_pos = X + emb_posicion(posiciones)           # SE SUMA. No se pega al lado.

with torch.no_grad():                     # no estamos entrenando: no hace falta anotar
    sal_orig2, _ = atencion(X_con_pos, W_q, W_k, W_v, causal=False)
    X_perm_con_pos = X[perm] + emb_posicion(posiciones)   # barajada, pero los sellos
    sal_perm2, _ = atencion(X_perm_con_pos, W_q, W_k, W_v, causal=False)  # en orden

dif = float((sal_perm2 - sal_orig2[perm]).abs().max())
print("diferencia máxima ahora:", round(dif, 4))
revisar(dif > 1e-3,
        "ahora las salidas SON distintas: el modelo puede notar que cambió el orden.",
        "siguen dando igual. Fijate que estés sumando emb_posicion a las dos.")

---

## 8. El bloque transformer completo

Cuatro piezas, y ninguna es nueva:

| pieza | qué hace | de dónde viene |
|---|---|---|
| **attention** | decide de dónde traer información | lo de hoy |
| **MLP** | decide qué hacer con lo que trajo | la suma de rampas de la clase 2 |
| **conexión residual** | deja pasar el original | «el expediente sigue adjunto» |
| **normalización** | que los números no se disparen | plomería |

La clase de abajo es **la misma que se usó para entrenar el modelo real** cuyos mapas vas
a mirar en la sección 9 (está en `medir-atencion.py`). Leela entera: son treinta líneas y
las entendés todas.

In [ ]:
class AtencionMulticabeza(nn.Module):
    """Cada palabra le pregunta a las anteriores qué tan relevantes son para ella."""

    def __init__(self, dim, cabezas, ctx):
        super().__init__()
        self.cabezas = cabezas
        self.d_cabeza = dim // cabezas          # el ancho se reparte entre las cabezas
        self.proy_q = nn.Linear(dim, dim, bias=False)       # "lo que busco"
        self.proy_k = nn.Linear(dim, dim, bias=False)       # "lo que ofrezco"
        self.proy_v = nn.Linear(dim, dim, bias=False)       # "lo que aporto"
        self.proy_salida = nn.Linear(dim, dim, bias=False)  # mezcla las cabezas al final
        # un buffer, no un parámetro: la máscara no se aprende, es una regla
        self.register_buffer("mascara", torch.tril(torch.ones(ctx, ctx)))

    def forward(self, x, devolver_pesos=False):
        B, T, D = x.shape                       # (lote, palabras, números)
        H, dc = self.cabezas, self.d_cabeza
        # una sola multiplicación para las 4 cabezas, y después se parte en 4
        q = self.proy_q(x).view(B, T, H, dc).transpose(1, 2)   # (B, H, T, dc)
        k = self.proy_k(x).view(B, T, H, dc).transpose(1, 2)
        v = self.proy_v(x).view(B, T, H, dc).transpose(1, 2)

        puntajes = (q @ k.transpose(-2, -1)) / dc ** 0.5       # (B, H, T, T)
        puntajes = puntajes.masked_fill(self.mascara[:T, :T] == 0, float("-inf"))
        pesos = torch.softmax(puntajes, dim=-1)                # filas que suman 1

        salida = pesos @ v                                     # (B, H, T, dc)
        salida = salida.transpose(1, 2).contiguous().view(B, T, D)   # pegar las cabezas
        salida = self.proy_salida(salida)
        return (salida, pesos) if devolver_pesos else (salida, None)


class BloqueTransformer(nn.Module):
    """attention + MLP, cada uno con su normalización y su conexión residual."""

    def __init__(self, dim, cabezas, ctx):
        super().__init__()
        self.norma1 = nn.LayerNorm(dim)
        self.atencion = AtencionMulticabeza(dim, cabezas, ctx)
        self.norma2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),   # se agranda: más rampas para dibujar
            nn.ReLU(),                 # LA rampa de la clase 2
            nn.Linear(4 * dim, dim),   # se vuelve al tamaño de siempre
        )

    def forward(self, x, devolver_pesos=False):
        a, pesos = self.atencion(self.norma1(x), devolver_pesos)
        x = x + a                          # conexión residual 1: el original sigue adjunto
        x = x + self.mlp(self.norma2(x))   # conexión residual 2
        return x, pesos


print("Listo. Un transformer entero, apagado.")

In [ ]:
# Verificación de formas: el hábito profesional de la clase.
DIM, CABEZAS, CTX = 96, 4, 24
bloque = BloqueTransformer(DIM, CABEZAS, CTX)

lote = torch.randn(2, 12, DIM)          # 2 oraciones de 12 palabras
salida_b, pesos_b = bloque(lote, devolver_pesos=True)

print(f"entra: {tuple(lote.shape)}   -> (oraciones, palabras, números)")
print(f"sale : {tuple(salida_b.shape)}   <- LA MISMA FORMA. Por eso se puede apilar.")
print(f"mapas de atención: {tuple(pesos_b.shape)}  -> (oraciones, cabezas, palabras, palabras)")
print()
print(f"el bloque tiene {utils.contar_parametros(bloque):,} perillas")

revisar(salida_b.shape == lote.shape,
        "entra y sale con la misma forma: la condición para poder apilar bloques.",
        "las formas no coinciden. Casi seguro es el .view() de las cabezas: "
        "fijate que dim sea divisible por cabezas.")
revisar(torch.allclose(pesos_b.sum(dim=-1), torch.ones(2, CABEZAS, 12), atol=1e-5),
        "las filas de las cuatro cabezas suman 1 en las dos oraciones.",
        "alguna fila no suma 1: revisá el dim del softmax.")

### `TODO 4` — sacale la conexión residual y mirá desvanecerse la señal

En la celda de abajo hay dos versiones del mismo apilado de seis bloques: una **con**
conexión residual y otra **sin** (le sacamos el `x +`). Miramos la desviación estándar de
la salida, que es una manera simple de preguntar «¿queda algo de señal acá adentro?».

**Pista.** Imprimí `salida.std()` en las dos y comparalas. Y si querés ver el efecto más
dramático, subí `N_BLOQUES` de 6 a 12.

**Cómo saber si te salió.** Sin residual, la desviación estándar de la salida se hace
mucho más chica que la de la entrada. Con residual, se mantiene en el mismo orden.

In [ ]:
N_BLOQUES = 6      # TODO: probá también con 12 y mirá cuánto empeora

entrada = torch.randn(1, 12, DIM)
print(f"la entrada tiene std = {entrada.std():.3f}")

# --- CON conexión residual (los bloques tal como están escritos) ---
x = entrada
for _ in range(N_BLOQUES):
    x, _ = BloqueTransformer(DIM, CABEZAS, CTX)(x)
print(f"con residual   ({N_BLOQUES} bloques): std = {x.std():.3f}")

# --- SIN conexión residual: la MISMA cuenta, pero sin el "x +" ---
y = entrada
for _ in range(N_BLOQUES):
    b = BloqueTransformer(DIM, CABEZAS, CTX)
    a, _ = b.atencion(b.norma1(y))
    y = a                       # <- acá estaría el x + a
    y = b.mlp(b.norma2(y))      # <- y acá el x + mlp(...)
print(f"sin residual   ({N_BLOQUES} bloques): std = {y.std():.3f}")

revisar(y.std() < x.std() / 3,
        "sin residual la señal se apagó. Por eso existe esa suma de una línea.",
        "esta vez no se apagó tanto: es ruido al azar, probá subiendo N_BLOQUES a 12.")

In [ ]:
# Apilar de verdad, y contar perillas.
class MiniTransformer(nn.Module):
    def __init__(self, vocab=3000, dim=96, cabezas=4, capas=6, ctx=24):
        super().__init__()
        self.emb_ficha = nn.Embedding(vocab, dim)      # una fila por palabra
        self.emb_posicion = nn.Embedding(ctx, dim)     # una fila por posición
        self.bloques = nn.ModuleList(
            [BloqueTransformer(dim, cabezas, ctx) for _ in range(capas)])
        self.norma_final = nn.LayerNorm(dim)
        self.cabeza = nn.Linear(dim, vocab, bias=False)

    def forward(self, idx):
        T = idx.shape[1]
        x = self.emb_ficha(idx) + self.emb_posicion(torch.arange(T))
        for bloque in self.bloques:
            x, _ = bloque(x)
        return self.cabeza(self.norma_final(x))


modelo = MiniTransformer()
fichas_prueba = torch.randint(0, 3000, (2, 12))
logits = modelo(fichas_prueba)

print("entra:", tuple(fichas_prueba.shape), "  (2 oraciones de 12 fichas)")
print("sale :", tuple(logits.shape), "  (un puntaje por cada palabra del vocabulario,")
print("        para cada posición: eso es «predecir la próxima ficha»)")
print()
print(f"perillas del transformer entero: {utils.contar_parametros(modelo):,}")
print(f"perillas de la red de la clase 4:  ~65")
print()
print("Es la misma máquina de dibujar de la clase 1. Con más perillas.")

---

## 9. Los mapas de un modelo entrenado de verdad

Todo lo de arriba usó matrices al azar. Ahora miremos qué hace un transformer que **sí**
se entrenó: el mismo bloque de la sección 8, entrenado sobre los discursos de apertura de
sesiones (1984-2019) con el script `medir-atencion.py` de esta carpeta.

Es el modelo que alimenta las cuatro visualizaciones de la clase. Los mapas están
guardados en `datos-atencion.json`, así que esto no entrena nada ni necesita internet.

In [ ]:
carpeta = utils.raiz_del_repo() / "clases" / "07-attention-y-transformer"
paquete = json.loads((carpeta / "datos-atencion.json").read_text(encoding="utf-8"))
meta = paquete["meta"]

print(f"modelo: {meta['parametros']:,} perillas · {meta['capas']} bloques · "
      f"{meta['cabezas']} cabezas · contexto {meta['contexto']}")
print(f"entrenado {meta['pasos']} pasos en {meta['segundos']:.0f} s de CPU")
print(f"loss de validación: {meta['loss_inicial']:.2f} -> {meta['loss_final']:.2f}")
print(f"vocabulario: {meta['vocabulario']:,} palabras "
      f"(cubre el {meta['cobertura']*100:.0f}% del corpus)")
print()
print("una muestra que genera:")
print(" ", meta["muestra_generada"])

In [ ]:
# Las cuatro cabezas de cada bloque, sobre la oración de la anáfora.
caso = paquete["oraciones"][0]
tokens = caso["fichas"]
print("oración (real, del corpus):", " ".join(tokens))
print("fenómeno:", caso["fenomeno"])

for capa in range(meta["capas"]):
    fig, ejes = g.figura(13, 11, filas=2, columnas=2,
                         titulo=f"Modelo ENTRENADO · bloque {capa + 1}, sus cuatro cabezas")
    for h, ax in enumerate(ejes.ravel()):
        info = paquete["cabezas"][capa][h]
        A = torch.tensor(caso["atencion"][capa][h])
        g.mapa_de_atencion(A, tokens, ax=ax, mascara_causal=True, anotar=False,
                           titulo=f"cabeza {h + 1}: {info['titulo']}")
    plt.show()

Mirá la diferencia entre los dos bloques.

En el **bloque 1** las cuatro cabezas dibujan casi la misma **diagonal**: cada palabra se
mira a sí misma y poco más. Todavía no hay mucho contexto que mezclar, y cuatro juegos de
perillas distintos terminaron en el mismo lugar. **No está roto: es lo que aprendió.**

En el **bloque 2** empieza a pasar algo: las cabezas se separan y algunas miran lejos. Y
ahí hay una que vale la pena mirar de cerca.

In [ ]:
# ¿A dónde mira la palabra «lo» en cada cabeza del bloque 2?
i_lo = tokens.index("lo")
for h in range(meta["cabezas"]):
    fila = caso["atencion"][1][h][i_lo]
    j = max(range(len(fila)), key=lambda k: fila[k])
    print(f"bloque 2, cabeza {h+1}: «lo» le presta {fila[j]*100:5.1f}% a «{tokens[j]}»")

cuantas = sum(1 for h in range(meta["cabezas"])
              if tokens[max(range(i_lo + 1),
                            key=lambda k: caso["atencion"][1][h][i_lo][k])] == "algo")
print()
print(f"En {cuantas} de las {meta['cabezas']} cabezas, «lo» se va derecho a «algo»,")
print("que es exactamente la palabra a la que «lo» se refiere. Nadie le enseñó")
print("gramática: el modelo aprendió a predecir la próxima palabra sobre discursos")
print("parlamentarios, y le convino mirar ahí.")
print()
print("Y ahora la parte incómoda, que es la del TODO 5:")
print("que le convenga mirar ahí NO prueba que «algo» sea la causa de lo que el")
print("modelo predice después.")

for h, info in enumerate(paquete["cabezas"][1]):
    e = info["est"]
    print(f"\ncabeza {h+1}: {info['titulo']}")
    print(f"   a la anterior {e['anterior']:.2f} · a la primera {e['primera']:.2f} "
          f"· a sí misma {e['misma']:.2f}")

### `TODO 5` (el ejercicio de humildad epistémica)

Elegí **una** de esas cuatro cabezas y hacé dos cosas:

1. Escribí una **hipótesis** de qué está haciendo. Una oración. Por ejemplo: *«la cabeza 2
   conecta cada palabra con la anterior, así que parece estar armando pares»*.

2. Escribí por qué esa hipótesis **no es una explicación** del comportamiento del modelo.

**Pista para la parte 2.** Se pueden construir mapas de atención muy distintos que
producen exactamente **la misma salida**. Si el mapa no determina la salida, entonces
mirar el mapa no alcanza para explicar la salida. Es la misma distinción que ya usás todos
los días: **correlación no es causalidad**. Que el modelo haya *mirado* una palabra no
prueba que esa palabra *haya causado* lo que dijo.

**Qué evidencia sí serviría.** Una **intervención**: cambiar esa palabra por otra y ver si
cambia la predicción. Eso es un experimento; el mapa es una descripción.

**Tu hipótesis:** _(escribí acá)_

**Por qué no es una explicación:** _(escribí acá)_

---

## Tabla de formas, para tener a mano

Cuando algo explote, el 80% de las veces es una forma. Esta tabla es la referencia
(`B` = oraciones del lote, `T` = palabras, `D` = números por palabra, `H` = cabezas).

| objeto | forma | se lee |
|---|---|---|
| `X` (una oración) | `(T, D)` | una fila por palabra |
| `x` (un lote) | `(B, T, D)` | varias oraciones juntas |
| `Q`, `K`, `V` | `(B, H, T, D/H)` | el ancho repartido entre las cabezas |
| `puntajes` | `(B, H, T, T)` | **cuadrada**: quién pregunta × a quién mira |
| `pesos` | `(B, H, T, T)` | lo mismo, pero cada fila suma 1 |
| salida de attention | `(B, T, D)` | igual que la entrada |
| salida del bloque | `(B, T, D)` | **igual que la entrada**: por eso se apila |
| `logits` | `(B, T, vocabulario)` | un puntaje por palabra posible, en cada posición |

Si una de estas no te da, imprimí las formas de a una hacia atrás hasta encontrar dónde se
rompió. No lo adivines: `print(algo.shape)` es más rápido que pensarlo.

---

## Qué te llevás

**Las tres cosas de hoy:**

1. **Attention es un promedio ponderado.** Cada palabra reparte su 100% entre las demás y
   se arma su propio resumen del contexto. Todo lo demás es plomería alrededor de esa
   cuenta.
2. **Los pesos se aprenden.** Query, key y value son tres transformaciones de la misma
   palabra, y sus matrices son perillas que ajusta el mismo descenso por gradiente de la
   clase 3. La atención no está en el texto: sale de los datos.
3. **Un mapa de atención muestra por dónde pasó la información, no por qué el modelo
   decidió.** Es la advertencia más útil que te llevás hoy, y sirve para leer papers
   ajenos y para escribir los propios.

**El vocabulario nuevo** (también está en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *attention* | cada elemento arma un resumen ponderado de los demás |
| *self-attention* | la oración se mira a sí misma; no hay dos textos |
| *query* | lo que una palabra busca |
| *key* | el aviso que una palabra publica |
| *value* | lo que una palabra aporta cuando la miran |
| *softmax* | puntajes convertidos en porcentajes que suman 100 |
| *causal mask* | tapar el futuro para que el modelo no copie la respuesta |
| *multi-head* | varias cabezas en paralelo, con perillas distintas |
| *positional encoding* | el sellito con el número de orden |
| *residual connection* | el original del expediente, siempre adjunto |
| *layer norm* | plomería para que los números no se disparen |

**Para el lunes, en tu propia investigación.** Si te cruzás con un paper que muestra un
mapa de atención y concluye *«el modelo se basa en la palabra X»*, ahí tenés un comentario
de revisor listo: eso es flujo de información, no explicación. Pedí una **intervención**
(cambiar la palabra y ver si cambia la predicción). Y si los mapas los mostrás vos,
reportalos como **exploratorios**.

**Lo que queda prometido.** Tenés un transformer entero y apagado. La clase 8 lo enciende:
le damos de comer cuarenta años de discursos y lo vemos aprender a escribir, de sopa de
letras a cadencia parlamentaria. Y ahí el softmax de hoy vuelve con otro nombre —
**temperatura** — y el `√d` y la ventana de contexto dejan de ser detalles.

---

### Para seguir

Las referencias están en `referencias.md`, con el orden en que conviene leerlas. La
primera recomendación es que el material visual sobre transformers lo leas **después** de
esta clase, no antes: ahora tenés dónde colgarlo.

### Las visualizaciones de esta clase

Están en `clases/07-attention-y-transformer/viz/` y abren con doble clic, sin internet:

- `attention-como-mesa.html` — hacé clic en una palabra y mirá su mapa.
- `qkv.html` — girá las perillas y mirá cambiar a quién mira cada palabra.
- `matriz-de-atencion.html` — **la que conviene volver a abrir** si lo de filas y columnas
  quedó flojo.
- `cabezas.html` — cuatro cabezas leyendo la misma oración, y una que no se entiende.
- `supuestos-de-arquitectura.html` — la tabla de la clase 5, con las dos filas grises ya
  abiertas.